In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
import torch

from utils import *
from darts_models import *
from seasonal_drift_model import *
from SIR_EAKF_model import *
# from SIR_AH_EAKF_model import *


c:\Users\ry2460\anaconda3\envs\timeseries_forecasting\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU available: True


In [2]:
season = 2026 
epiyear = 2025 
epiweek = 47
ref_date = epiweek_to_dates(epiyear, epiweek).enddate() #Saturday at the end of epiweek
ref_date = pd.Timestamp(ref_date)

data_dir = '../data/'
results_dir = '../results/'
figures_dir = f'../figures/{season}' 

locations_fname = data_dir +"locations.csv"
locations = pd.read_csv(locations_fname)
loc_name2abbr = dict(zip(locations['location_name'], locations['abbreviation']))
loc_name2loc = dict(zip(locations['location_name'], locations['location']))
locations = locations.set_index('abbreviation')
abbr2loc = dict(zip(locations.index, locations['location']))
loc2abbr = dict(zip(locations['location'],locations.index))
states = locations.index.values
num_states = len(states)

populations_fname = data_dir +"populations.csv"
populations = pd.read_csv(populations_fname)
# df_pop = generate_pop_per_week(states, populations)

# AH_daily, df_AH = read_AH(data_dir)

num_samples = 1000
alpha_vals = [0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5 ,0.6, 0.7, 0.8, 0.9]
quantiles = np.append(np.append([0.01,0.025],np.arange(0.05,0.95+0.05,0.050)),[0.975,0.99])
weeks_to_predict = 4

In [3]:
fix_partial_reporting=False
fix_outliers=False
df_hosp = read_hosp_incidence_data(data_dir, epiyear, epiweek, states, 
                                   new_format=True, download=False,
                                   fix_partial_reporting=fix_partial_reporting, 
                                   fix_outliers=fix_outliers,
                                   plot=False)
df_hosp = df_hosp[df_hosp.date<pd.to_datetime(ref_date)]


In [18]:
ensemble_models = ["ExponentialSmoothing_mod_ili", #"ExponentialSmoothing", 
                   "LightGBM", "LightGBM_mod_ili", 
                   "seasonal_drift_mod_ili", #"seasonal_drift", "seasonal_drift_mod_ili_pool0.5",
                   "SIR-EAKF_start_250809"] #,"SIR-EAKF_start_250906"],"SIR-EAKF_start_251004"]

ensemble_dict = {}
# load models predictions
for model in ensemble_models:
    print("-----------Loading model: {}-----------".format(model))
    df_results = load_pred_result_files(results_dir, model, locations)
    ensemble_dict[model] = df_results

mean_ensemble_name = 'Mean_CU_Ensemble'
weighted_ensemble_name = 'WIS_Weighted_CU_Ensemble'

#replace start date: start_date='2025-10-04' #'2025-11-22'
print("-----------Generating mean ensemble: {}-----------".format(mean_ensemble_name))
generate_mean_ensemble_pred_results(ensemble_dict, results_dir, mean_ensemble_name)
df_metrics_models = calc_models_pred_fit(df_hosp, ensemble_models, season, locations, 
                                  results_dir, figures_dir, alpha_vals, plot=False)
df_weights = generate_pred_weights(ensemble_dict, df_metrics_models, locations, decay_rate=1.0)
print("-----------Generating weighted ensemble: {}-----------".format(weighted_ensemble_name))
generate_weighted_pred_results(ensemble_dict, df_weights, locations, results_dir, weighted_ensemble_name) 

df_metrics_ensembles = calc_models_pred_fit(df_hosp, [mean_ensemble_name,weighted_ensemble_name],season,
                                            locations, results_dir, figures_dir, alpha_vals, plot=False)

df_metrics = pd.concat([df_metrics_models, df_metrics_ensembles], ignore_index=True)
df_metrics.to_csv(results_dir+"/_metrics/metrics_" +ref_date.strftime("%Y-%m-%d") +'.csv',index=False)

-----------Loading model: ExponentialSmoothing_mod_ili-----------
-----------Loading model: LightGBM-----------
-----------Loading model: LightGBM_mod_ili-----------
-----------Loading model: seasonal_drift_mod_ili-----------
-----------Loading model: SIR-EAKF_start_250809-----------
-----------Generating mean ensemble: Mean_CU_Ensemble-----------
-----------Calculating metrics for model: ExponentialSmoothing_mod_ili-----------
-----------Calculating metrics for model: LightGBM-----------
-----------Calculating metrics for model: LightGBM_mod_ili-----------
-----------Calculating metrics for model: seasonal_drift_mod_ili-----------
-----------Calculating metrics for model: SIR-EAKF_start_250809-----------
-----------Generating weighted ensemble: WIS_Weighted_CU_Ensemble-----------
-----------Calculating metrics for model: Mean_CU_Ensemble-----------
-----------Calculating metrics for model: WIS_Weighted_CU_Ensemble-----------


In [19]:
df_metrics = pd.read_csv(results_dir+"/_metrics/metrics_" +ref_date.strftime("%Y-%m-%d") +'.csv')
additional_model = ["FluSight-baseline"] #["LightGBM_mod_ili"] #["ExponentialSmoothing_mod_ili"] 
df_metrics = df_metrics[~df_metrics['model'].isin(additional_model)]
df_metrics_additional = calc_models_pred_fit(df_hosp, additional_model, season, locations, 
                                             results_dir, figures_dir, alpha_vals, plot=False)
df_metrics = pd.concat([df_metrics, df_metrics_additional], ignore_index=True) 
df_metrics.to_csv(results_dir+"/_metrics/metrics_" +ref_date.strftime("%Y-%m-%d") +'.csv',index=False)

-----------Calculating metrics for model: FluSight-baseline-----------


In [ ]:
# models_to_plot = ensemble_models + [mean_ensemble_name,weighted_ensemble_name]
models_to_plot = [mean_ensemble_name,weighted_ensemble_name] 
# models_to_plot = "LightGBM", "LightGBM_mod_ili"] 
# models_to_plot = ["SIR-EAKF_start_250906"]
calc_models_pred_fit(df_hosp, models_to_plot, season, locations, 
                     results_dir, figures_dir, alpha_vals, plot=True)

In [47]:
def harmonize_output(df):
    df = df.copy()

    # datetimes
    df["reference_date"] = pd.to_datetime(df["reference_date"], errors="coerce")
    df["target_end_date"] = pd.to_datetime(df["target_end_date"], errors="coerce")

    # strings
    df["target"] = df["target"].astype("string")
    df["location"] = df["location"].astype("string")
    df["output_type"] = df["output_type"].astype("string")
    df["output_type_id"] = df["output_type_id"].astype("string")

    # numeric
    # if you want strict integers with missing allowed:
    df["horizon"] = df["horizon"].astype("Int64")   # nullable integer
    df["value"] = df["value"].astype("float64")

    return df

In [58]:
nowcast_file = f"{data_dir}nowcast_{epiyear}_{epiweek}.csv"
df_nowcast_pred = load_and_format_nowcast_pred(nowcast_file, ref_date, loc_name2loc)
df_nowcast_pred = harmonize_output(df_nowcast_pred)
# df_nowcast_pred.info()
df_nowcast_pred

,reference_date,target,horizon,target_end_date,location,output_type,output_type_id,value
0,2025-11-22,wk inc flu hosp,-1,2025-11-15,01,quantile,0.01,21.0
1,2025-11-22,wk inc flu hosp,-1,2025-11-15,01,quantile,0.025,21.0
2,2025-11-22,wk inc flu hosp,-1,2025-11-15,01,quantile,0.05,21.0
3,2025-11-22,wk inc flu hosp,-1,2025-11-15,01,quantile,0.1,21.0
4,2025-11-22,wk inc flu hosp,-1,2025-11-15,01,quantile,0.15,21.0
...,...,...,...,...,...,...,...,...
1214,2025-11-22,wk inc flu hosp,-1,2025-11-15,56,quantile,0.85,7.0
1215,2025-11-22,wk inc flu hosp,-1,2025-11-15,56,quantile,0.9,9.0
1216,2025-11-22,wk inc flu hosp,-1,2025-11-15,56,quantile,0.95,10.0
1217,2025-11-22,wk inc flu hosp,-1,2025-11-15,56,quantile,0.975,12.0


In [53]:
df_peak_pred = pd.read_csv(f"{results_dir}peak_forecasts/{format(ref_date,'%Y-%m-%d')}.csv")
df_peak_pred = parse_date_column(df_peak_pred,"reference_date")
df_peak_pred = harmonize_output(df_peak_pred)
# df_peak_pred.info()
df_peak_pred

,reference_date,target,horizon,target_end_date,location,output_type,output_type_id,value
0,2025-11-22,peak week inc flu hosp,<NA>,NaT,01,pmf,2025-11-22,0.000700
1,2025-11-22,peak week inc flu hosp,<NA>,NaT,01,pmf,2025-11-29,0.004900
2,2025-11-22,peak week inc flu hosp,<NA>,NaT,01,pmf,2025-12-06,0.010300
3,2025-11-22,peak week inc flu hosp,<NA>,NaT,01,pmf,2025-12-13,0.029000
4,2025-11-22,peak week inc flu hosp,<NA>,NaT,01,pmf,2025-12-20,0.069900
...,...,...,...,...,...,...,...,...
2592,2025-11-22,peak inc flu hosp,<NA>,NaT,US,quantile,0.85,49876.460211
2593,2025-11-22,peak inc flu hosp,<NA>,NaT,US,quantile,0.9,54696.602063
2594,2025-11-22,peak inc flu hosp,<NA>,NaT,US,quantile,0.95,61873.274077
2595,2025-11-22,peak inc flu hosp,<NA>,NaT,US,quantile,0.975,69078.977730


In [54]:
final_model = weighted_ensemble_name
df_weekly_pred = load_pred_result_files(results_dir, final_model, locations)
df_weekly_pred = df_weekly_pred[df_weekly_pred['reference_date']==ref_date]
df_weekly_pred = harmonize_output(df_weekly_pred)
# df_weekly_pred.info()
df_weekly_pred

,reference_date,target,horizon,target_end_date,location,output_type,output_type_id,value
0,2025-11-22,wk inc flu hosp,0,2025-11-22,US,quantile,0.01,766.000
1,2025-11-22,wk inc flu hosp,0,2025-11-22,US,quantile,0.025,948.000
2,2025-11-22,wk inc flu hosp,0,2025-11-22,US,quantile,0.05,1126.000
3,2025-11-22,wk inc flu hosp,0,2025-11-22,US,quantile,0.1,1396.000
4,2025-11-22,wk inc flu hosp,0,2025-11-22,US,quantile,0.15,1620.000
...,...,...,...,...,...,...,...,...
5931,2025-11-22,wk flu hosp rate change,3,2025-12-13,72,pmf,stable,0.137
5932,2025-11-22,wk flu hosp rate change,3,2025-12-13,72,pmf,increase,0.265
5933,2025-11-22,wk flu hosp rate change,3,2025-12-13,72,pmf,large_increase,0.303
5934,2025-11-22,wk flu hosp rate change,3,2025-12-13,72,pmf,decrease,0.294


In [59]:
df_all = pd.concat([df_weekly_pred, df_nowcast_pred], ignore_index=True)
df_all = df_all.sort_values(
    by=["target", "location", "horizon"],
    ascending=[False, True, True],
    kind="mergesort"   # stable sort, preserves order within ties if you care
).reset_index(drop=True)
df_all = pd.concat([df_all, df_peak_pred], ignore_index=True)
df_all.to_csv(f"{results_dir}CU-ensemble/{format(ref_date,'%Y-%m-%d')}-CU-ensemble.csv", index=False)